### page6.chapter1.[실습] Tool 여러 개 준비하기

In [3]:
from langchain_core.tools import tool
import ast, operator

# [1번 문제]
# 도시 이름을 받아 날씨를 반환하는 get_weather(city: str) 함수를
# @tool 데코레이터와 함께 작성하세요. (docstring 필수!)
@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 날씨 정보를 반환합니다."""
    weather_db = {"서울": "맑음, 23도"}
    return weather_db.get(city, "정보 없음")

# [2번 문제]
# 사칙연산 문자열(예: "3 + 5")을 계산해 반환하는
# calculate(expression: str) 함수를 @tool로 작성하세요.
# (ast.parse와 ast.BinOp를 활용하면 eval보다 안전하게 계산할 수 있습니다)
# eval("5+6") --> 파이썬 코드로 실행해줌 "a=10 for in dl  "
_OPS = {
    ast.Add: operator.add,      # + 기호
    ast.Sub: operator.sub,      # - 기호
    ast.Mult: operator.mul,     # * 기호
    ast.Div: operator.truediv,  # / 기호
    ast.USub: operator.neg,     # 음수 부호 (-)
    ast.UAdd: operator.pos      # 양수 부호 (+)
}
def _safe_eval(node): # 함수명 앞에 _를 붙인건 private의 의미임
    if isinstance(node, ast.Constant):# 숫자인지?
        return node.value
    if isinstance(node, ast.BinOp):# Binary Operation 이항연산자인가?
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):# 단항연산자인가?
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("허용되지 않은 수식입니다.")

@tool
def calculate(expression: str) -> str:
    """사칙연산 수식 문자열을 계산합니다. 예: '3 + 5 * 2'"""
    try:# mode를 eval로 지정하면 expression문자열에 단일 수식만 허용하고 
        tree = ast.parse(expression, mode="eval") # 결과물은 Tree 객체로 반환(연산자 우선순위로 정리된 계층구조)
        result = _safe_eval(tree.body) # Tree 안에 값을 꺼내서 함수로 전달
        return f"{expression} = {result}"
    except Exception:
        return "계산할 수 없는 수식입니다."

# [도전 문제]
# get_weather.invoke({"city": "부산"}) 와
# calculate.invoke({"expression": "10 * (2 + 3)"}) 를 각각 실행해
# Tool 함수가 Agent 없이도 단독으로 잘 동작하는지 먼저 확인하세요.
print(get_weather.invoke({"city": "서울"}))
print(calculate.invoke({"expression": "10 * (2 + 3)"}))




맑음, 23도
10 * (2 + 3) = 50


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")  # 환경변수에서 꺼내쓰기

llm = ChatOpenAI(
    model = "gpt-4o-mini",
    api_key = api_key
)

# agent 객체 생성
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="너는 도움이 되는 비서야.",
)


C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "서울 날씨 어때?"}
    ]
})
# result값의 구조{"messages":[HumanMessage,AIMessage,ToolMessage,AIMessage]}
print(result["messages"][1].tool_calls)
final = result["messages"][-1]
print(final.content)


[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_6Mvbw3pPZeZHSx9zbuyZ9GBq', 'type': 'tool_call'}]
서울의 날씨는 맑고 기온은 23도입니다.


In [9]:
from langchain.agents import create_agent

# [1번 문제]
# 앞 슬라이드에서 만든 get_weather 하나만 tools로 갖는 agent를
# create_agent(model=llm, tools=[...], system_prompt="...") 로 만드세요.
agent = create_agent(
    model = llm,
    tools = [get_weather],
    system_prompt="너는 도움이 되는 비서야."
)
# [2번 문제]
# agent.invoke({"messages": [{"role": "user", "content": "대구 날씨 어때?"}]})
# 를 실행하고, result["messages"][-1].content를 출력하세요.
result = agent.invoke(
    {"messages":[{"role":"user","content":"서울 날씨는 어때?"}]}
)
print(result["messages"][-1].content)
# [도전 문제]
# result["messages"] 를 순서대로 출력해보며, HumanMessage ->
# AIMessage(tool_calls) -> ToolMessage -> AIMessage(최종 답변) 흐름이
# 실제로 몇 개의 메시지로 이루어지는지 세어보세요.
print(result["messages"])


서울의 날씨는 맑고, 기온은 23도입니다.
[HumanMessage(content='서울 날씨는 어때?', additional_kwargs={}, response_metadata={}, id='cfa311fb-ab26-4a6e-b1ab-fca0317f8cfa'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 62, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b11ef98e16', 'id': 'chatcmpl-Ed6JOaZtb9DeYH58Rgbs8IJVzbGxA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a17666-0b55-7690-9ad0-65e141c2e5a9-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_PciStezaIq7RlIikFS5VOdq4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_token

### page12

In [ ]:
# 앞 슬라이드의 get_weather, calculate 두 Tool을 모두 사용합니다.

# [1번 문제]
# [get_weather, calculate] 두 Tool을 가진 agent를 만드세요.
agent = create_agent(
    model = llm,
    tools = [get_weather,calculate],
    system_prompt="필요할 때만 도구를 써"
    # system_prompt="해당 정보가 없으면 니가 알아서 답변해줘"
)
# [2번 문제]
# "대구 날씨 알려줘" 와 "7 곱하기 8은 얼마야?" 두 질문을 각각
# invoke()해서, 서로 다른 도구가 호출되는지 답변으로 확인하세요.
results = agent.invoke({
    "messages":{"role":"user","content":"대구 날씨 알려줘"}
})

print(results["messages"][-1].content)

# [도전 문제]
# "안녕하세요!" 처럼 두 도구 모두와 관련 없는 질문도 invoke()해보고,
# result["messages"] 안에 ToolMessage가 있는지 없는지 확인해
# Agent가 도구를 쓰지 않고 바로 답했는지 판단해보세요.
results = agent.invoke({
        "messages":{"role":"user","content":"안녕하세요!!"}
    }
)
print(results["messages"][1]) # AIMessage에 tool_calls=[] 비어있으면 안쓰고 있음
print(results["messages"][-1].content)


content='안녕하세요! 어떻게 도와드릴까요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 96, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9fdb32c1c4', 'id': 'chatcmpl-Ed6a9aStUhjdvGETKyZ3UxJQhyVO6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a17675-e767-79c2-a6b5-84836f2b1278-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 96, 'output_tokens': 11, 'total_tokens': 107, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
안녕하세요! 어떻게 도와드릴까요?


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

@tool
def search_notice(query: str) -> str:
    """공지사항에서 관련 내용을 검색합니다."""
    docs = retriever.invoke(query)
    return "\n".join(
        d.page_content for d in docs)


### page13

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

#환경변수 읽기
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")  # 환경변수에서 꺼내쓰기

#LLM 연결하기
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    api_key = api_key
)

# 벡터 변환을 위한 임베딩 객체 준비
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 임시 데이터(문서로부터 가져오게 되는 값)
#  --> 청크로 나눴다는 가정하고
notice_chunks = [
    "[공지] 2026년 11월 1일(일) 02:00부터 06:00까지 시스템 안정화를 위한 정기 서버 점검이 진행됩니다.",
    "[영향도] 점검 시간 동안 로그인, 결제 서비스, 마이페이지 등 주요 기능의 이용이 일시 중단됩니다.",
    "[주의사항] 데이터 유실 방지를 위해 점검 시작 전 진행 중인 작업 및 결제는 미리 완료/저장해 주시기 바랍니다.",
    "[문의] 점검 완료 후 즉시 정상화될 예정이며, 관련 문의는 고객센터(1588-0000) 또는 1:1 문의를 이용해 주세요."
]
